# DC1 – Emulating Global Ocean: Submission Workflow

This notebook guides you through the **complete submission pipeline** for the
[DC1 Ocean Emulation Challenge](https://dc1-emulating-global-ocean.readthedocs.io/):

1. **Configure** your submission metadata and paths
2. **Prepare** your model predictions  *(customisable block)*
3. **Inspect** the dataset structure
4. **Validate** conformity with the DC1 grid specification
5. **Evaluate** against observations using the full DC1 pipeline
6. **Analyse** and visualise the results
7. **Submit** to the public leaderboard

> **DC1 is surface-only.** Your predictions must cover the global ocean at
> 0.25° resolution (lat × lon) with 10 daily lead times per forecast date.
> Depth is always the surface level (0.494 m).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-ai-data-challenges/dc1-emulating-global-ocean/blob/main/notebooks/submit.ipynb)


## 0. Setup

If running on **Colab**, install the project first:

```bash
!pip install git+https://github.com/ocean-ai-data-challenges/dc1-emulating-global-ocean.git
```

Otherwise, ensure the `dc-env` conda environment is activated  
(see the [Quickstart guide](https://dc1-emulating-global-ocean.readthedocs.io/en/latest/content/quickstart.html)).


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Engine 'argo' loading failed", category=RuntimeWarning)

import copy
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# Ensure the project root is on the Python path (works from notebooks/)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. Configuration

Set the metadata and paths for your submission here. This is the **only cell
you must edit** before running the notebook end-to-end.


In [ ]:
# ── Submission metadata ──────────────────────────────────────────────
MODEL_NAME   = "MyModel"                 # Short model identifier
TEAM_NAME    = "My Team"                 # Team / author name
DESCRIPTION  = "Short model description"

# ── Paths ────────────────────────────────────────────────────────────
PREDICTION_DIR = Path("/tmp/dc1_submission")   # Where your predictions live
OUTPUT_DIR     = PROJECT_ROOT / "dc1_output"   # Evaluation output directory


---

## 2. Prepare your predictions  ✉️

**This is the block you should customise.** Replace the example below with
the code that loads or generates your model's surface predictions.

### Requirements

DC1 predictions are **2-D surface fields** on a global 0.25° grid:

| Dimension | Range | Step | Points |
|-----------|-------|------|--------|
| Latitude  | −78° to 90° | 0.25° | 673 |
| Longitude | −180° to 180° | 0.25° | 1 440 |
| Time      | 10 daily lead times per forecast | 1 day | 10 |

**Variables**: `zos` (SSH), `thetao` (SST), `so` (SSS), `uo` / `vo` (surface velocities).

**Accepted layouts**:
- A directory of per-date `.zarr` stores (e.g. `20240101.zarr`, `20240108.zarr`, …)
- A single `.zarr` or `.nc` file covering the full period
- A directory of per-date `.nc` files

The evaluation period is **2024-01-01 to 2025-01-01** with one forecast every **7 days** (52 init dates).

> **Note:** No `depth` dimension is required — DC1 is surface-only.
> If your model produces 3-D output, slice the surface level (`depth ≈ 0.49 m`)
> before saving.


In [ ]:
# =====================================================================
# ✏️  CUSTOMISE THIS CELL
#
# Replace the sample generator with your own data-loading code.
# The result should be a directory of Zarr/NetCDF forecast files
# written to PREDICTION_DIR.
# =====================================================================

# ─── Example 1: Generate random-noise sample (for testing) ───────────
DC1_LAT       = np.arange(-78.0, 90.25, 0.25)
DC1_LON       = np.arange(-180.0, 180.0, 0.25)
DC1_VARIABLES = ["zos", "thetao", "so", "uo", "vo"]
N_LEAD_TIMES  = 10

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
init_dates = pd.date_range("2024-01-01", "2025-01-01", freq="7D")

for init_date in init_dates:
    ds = xr.Dataset(
        {var: xr.DataArray(
            rng.standard_normal((N_LEAD_TIMES, len(DC1_LAT), len(DC1_LON))).astype("float32"),
            dims=["time", "lat", "lon"],
        ) for var in DC1_VARIABLES},
        coords={"time": np.arange(N_LEAD_TIMES), "lat": DC1_LAT, "lon": DC1_LON},
    )
    ds.to_zarr(str(PREDICTION_DIR / f"{init_date:%Y%m%d}.zarr"), mode="w", consolidated=True)

print(f"Generated {len(init_dates)} forecast files in {PREDICTION_DIR}")

# ─── Example 2: Load from local Zarr stores ─────────────────────────
#
#   PREDICTION_DIR = Path("/data/my_model/forecasts")
#   # Should contain files like: 20240101.zarr/  20240108.zarr/  ...

# ─── Example 3: Load from a single NetCDF file ──────────────────────
#
#   ds = xr.open_dataset("/data/my_model/predictions.nc")
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   for date, group in ds.groupby("time.date"):
#       store_path = PREDICTION_DIR / f"{pd.Timestamp(date):%Y%m%d}.zarr"
#       group.to_zarr(str(store_path), mode="w", consolidated=True)

# ─── Example 4: Download from S3 (Wasabi / AWS) ─────────────────────
#
#   import s3fs
#   fs = s3fs.S3FileSystem(
#       key="YOUR_KEY",
#       secret="YOUR_SECRET",
#       client_kwargs={"endpoint_url": "https://s3.eu-west-2.wasabisys.com"},
#   )
#   PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
#   for remote in fs.ls("my-bucket/DC1/MyModel/"):
#       fs.get(remote, str(PREDICTION_DIR / Path(remote).name), recursive=True)

# ─── Example 5: Download from Copernicus Marine (surface slice) ──────
#
#   import copernicusmarine as cmc
#   ds = cmc.open_dataset(
#       dataset_id="cmems_mod_glo_phy-all_anfc_0.083deg_P1D-m",
#       variables=["zos", "thetao", "so", "uo", "vo"],
#       minimum_depth=0, maximum_depth=1,
#       minimum_longitude=-180, maximum_longitude=180,
#       minimum_latitude=-78, maximum_latitude=90,
#       start_datetime="2024-01-01", end_datetime="2024-01-11",
#   )
#   # Regrid to 0.25° and save as Zarr stores


## 3. Inspect the dataset

Open one of the forecast files and verify its structure.


In [ ]:
pred_files = sorted(PREDICTION_DIR.glob("*.zarr")) or sorted(PREDICTION_DIR.glob("*.nc"))

print(f"Found {len(pred_files)} forecast files in {PREDICTION_DIR}:")
for f in pred_files[:10]:
    print(f"  {f.name}")
if len(pred_files) > 10:
    print(f"  ... and {len(pred_files) - 10} more")


In [ ]:
f0 = pred_files[0]
ds0 = xr.open_zarr(str(f0)) if (f0.suffix == ".zarr" or f0.is_dir()) else xr.open_dataset(str(f0))

print(f"Latitude:   {ds0.lat.values[0]:.2f} to {ds0.lat.values[-1]:.2f}  ({len(ds0.lat)} pts, step {np.diff(ds0.lat.values[:2])[0]:.2f}°)")
print(f"Longitude:  {ds0.lon.values[0]:.2f} to {ds0.lon.values[-1]:.2f}  ({len(ds0.lon)} pts, step {np.diff(ds0.lon.values[:2])[0]:.2f}°)")
print(f"Time:       {ds0.time.values[0]} to {ds0.time.values[-1]}  ({len(ds0.time)} lead times)")
print(f"Variables:  {list(ds0.data_vars)}")
ds0


## 4. Validate against the DC1 specification

Check grid conformity, variable presence, temporal coverage, and NaN fraction
before running the (expensive) full evaluation.


In [ ]:
# ── DC1 specification constants ───────────────────────────────────────
_DC1_LAT         = np.arange(-78.0, 90.25, 0.25)
_DC1_LON         = np.arange(-180.0, 180.0, 0.25)
_DC1_VARIABLES   = ["zos", "thetao", "so", "uo", "vo"]
_DC1_START       = pd.Timestamp("2024-01-01")
_DC1_END         = pd.Timestamp("2025-01-01")
_N_LEAD_TIMES    = 10
_N_DAYS_INTERVAL = 7
_MAX_NAN_FRAC    = 0.10


def validate_submission(prediction_dir: Path) -> list[str]:
    """Run DC1 conformity checks. Returns a list of issues (empty = all good)."""
    issues = []

    files = sorted(prediction_dir.glob("*.zarr")) or sorted(prediction_dir.glob("*.nc"))
    if not files:
        issues.append("❌ No .zarr or .nc files found in prediction directory.")
        return issues

    # Temporal coverage
    expected_dates = pd.date_range(_DC1_START, _DC1_END, freq=f"{_N_DAYS_INTERVAL}D")
    print(f"Expected {len(expected_dates)} init dates, found {len(files)} files.")
    if len(files) < len(expected_dates):
        issues.append(f"⚠️  Expected {len(expected_dates)} files, found {len(files)}.")

    # First-file detailed check
    f0  = files[0]
    ds0 = xr.open_zarr(str(f0)) if (f0.suffix == ".zarr" or f0.is_dir()) else xr.open_dataset(str(f0))

    # Variables
    missing = set(_DC1_VARIABLES) - set(ds0.data_vars)
    if missing:
        issues.append(f"❌ Missing variables: {sorted(missing)}")
    else:
        print(f"✅ All {len(_DC1_VARIABLES)} required variables present.")

    # Latitude
    if len(ds0.lat) != len(_DC1_LAT):
        issues.append(f"❌ Latitude: expected {len(_DC1_LAT)} pts, got {len(ds0.lat)}.")
    elif not np.allclose(ds0.lat.values, _DC1_LAT, atol=0.01):
        issues.append("❌ Latitude values do not match the DC1 grid.")
    else:
        print("✅ Latitude grid matches DC1 specification.")

    # Longitude
    if len(ds0.lon) != len(_DC1_LON):
        issues.append(f"❌ Longitude: expected {len(_DC1_LON)} pts, got {len(ds0.lon)}.")
    elif not np.allclose(ds0.lon.values, _DC1_LON, atol=0.01):
        issues.append("❌ Longitude values do not match the DC1 grid.")
    else:
        print("✅ Longitude grid matches DC1 specification.")

    # Lead times
    if len(ds0.time) != _N_LEAD_TIMES:
        issues.append(f"❌ Time dimension: expected {_N_LEAD_TIMES} lead times, got {len(ds0.time)}.")
    else:
        print(f"✅ Time dimension has {_N_LEAD_TIMES} lead times.")

    # NaN fraction (first file)
    for var in _DC1_VARIABLES:
        if var in ds0.data_vars:
            nan_frac = float(ds0[var].isnull().mean().values)
            status   = "✅" if nan_frac <= _MAX_NAN_FRAC else "❌"
            print(f"  {status} {var}: NaN fraction = {nan_frac:.2%}")
            if nan_frac > _MAX_NAN_FRAC:
                issues.append(f"❌ Variable '{var}' has {nan_frac:.1%} NaN (max {_MAX_NAN_FRAC:.0%}).")

    ds0.close()
    return issues


issues = validate_submission(PREDICTION_DIR)

if issues:
    print(f"\n━━ {len(issues)} issue(s) found ━━")
    for issue in issues:
        print(f"  {issue}")
else:
    print("\n✅ All checks passed — your submission is DC1-compliant!")


## 5. Run the full evaluation pipeline

The DC1 evaluation compares your **surface predictions** against five independent
reference datasets: **Argo profiles**, **GLORYS12**, **Jason-3**, **SARAL**, and **SWOT**.

The pipeline is orchestrated by `dc1/evaluate.py`, which:
1. Downloads observation data from the Wasabi S3 bucket (cached in `dc1_output/`)
2. Aligns predictions with observations in space and time
3. Computes per-lead-time RMSE, RMSD, MAE, and bias metrics
4. Produces `results_<model>.json` and `results_<model>_per_bins.jsonl.gz`

> **Note:** The full evaluation downloads several GB of observation data and
> typically takes **1–2 h** on a 16-core / 30 GB machine.
> Ensure ~20 GB of free disk space and a stable internet connection.

### How to point the pipeline at your model

The cell below creates a custom YAML config that replaces the default GloNet
source entry with your local predictions (**Option B** — no S3 upload needed).

To upload your predictions to S3 and evaluate via **Option A** (recommended
for reproducibility), add a source entry in `dc1/config/dc1_wasabi.yaml`
pointing to your S3 path.


In [ ]:
import yaml

# Load the default DC1 config
config_path = PROJECT_ROOT / "dc1" / "config" / "dc1_wasabi.yaml"
with open(config_path) as f:
    dc1_config = yaml.safe_load(f)

model_name_lower = MODEL_NAME.lower()
custom_config    = copy.deepcopy(dc1_config)

# Replace the glonet source entry with a local-path source
for i, source in enumerate(custom_config.get("sources", [])):
    if source.get("dataset") == "glonet":
        custom_config["sources"][i] = {
            "dataset": model_name_lower,
            "observation_dataset": False,
            "config": "s3",
            "connection_type": "local",
            "local_root": str(PREDICTION_DIR.resolve()),
            "file_pattern": "**/*.zarr",
            "full_day_data": True,
            "ignore_geometry": True,
            "keep_variables": ["zos", "thetao", "so", "uo", "vo"],
            "eval_variables": ["zos", "thetao", "so", "uo", "vo"],
        }
        break

# Update dataset_references to use the custom model name
custom_config["dataset_references"] = {
    model_name_lower: ["argo_profiles", "glorys", "jason3", "saral", "swot"]
}

# Write the custom config
custom_config_path = PROJECT_ROOT / "dc1" / "config" / f"dc1_custom_{model_name_lower}.yaml"
with open(custom_config_path, "w") as f:
    yaml.dump(custom_config, f, default_flow_style=False, sort_keys=False)

print(f"Custom config written to: {custom_config_path}")
print(f"Model: {model_name_lower}")
print(f"Prediction path: {PREDICTION_DIR.resolve()}")


In [ ]:
# Launch the evaluation pipeline
# Change config_name to "dc1_wasabi" to evaluate the GloNet baseline instead.
config_name = f"dc1_custom_{model_name_lower}"

cmd = [
    sys.executable, str(PROJECT_ROOT / "dc1" / "evaluate.py"),
    "--config_name", config_name,
    "--data_directory", str(OUTPUT_DIR),
]

print(f"Running: {' '.join(cmd)}")
print("This may take a while (downloading observations + computing metrics)…\n")

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("\n━━ STDERR ━━")
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
else:
    print("\n✅ Evaluation completed successfully!")


## 6. Load and inspect results

The evaluation produces a JSON results file in `dc1_output/results/`.


In [ ]:
# Find the results file (own model → GloNet fallback → leaderboard copy)
results_file = OUTPUT_DIR / "results" / f"results_{model_name_lower}.json"
if not results_file.exists():
    results_file = OUTPUT_DIR / "results" / "results_glonet.json"
if not results_file.exists():
    results_file = PROJECT_ROOT / "dc1" / "leaderboard_results" / "results_glonet.json"

if results_file.exists():
    with open(results_file) as f:
        data = json.load(f)
    evaluated_model = data["dataset"]
    records = data["results"][evaluated_model]
    print(f"Results loaded for model: {evaluated_model}")
    print(f"Number of evaluation records: {len(records)}")
    print(f"Reference datasets: {sorted(set(r['ref_alias'] for r in records))}")
else:
    print("No results file found. Run the evaluation cell (section 5) first.")
    records, evaluated_model = [], MODEL_NAME.lower()


In [ ]:
# Parse scores grouped by reference dataset, variable and lead time
scores = {}  # {ref_alias: {variable: {lead_time: value}}}

for record in records:
    ref = record["ref_alias"]
    lt  = record.get("lead_time")
    for metric in record["result"]:
        if metric["Metric"] not in ("rmse", "rmsd"):
            continue
        var = metric["Variable"]
        scores.setdefault(ref, {}).setdefault(var, {})[lt] = metric["Value"]

# Print summary table
for ref_name in sorted(scores):
    print(f"\n━━ {ref_name.upper()} ━━")
    for var_name in sorted(scores[ref_name]):
        mean_val = np.mean(list(scores[ref_name][var_name].values()))
        print(f"  {var_name:40s}  mean RMSE = {mean_val:.4f}")


## 7. Visualise results


In [ ]:
import matplotlib.pyplot as plt

if "glorys" in scores:
    glorys = scores["glorys"]

    # All surface variables RMSE vs lead time
    fig, ax = plt.subplots(figsize=(10, 5))
    for var_name in sorted(glorys):
        lts  = sorted(glorys[var_name].keys())
        vals = [glorys[var_name][lt] for lt in lts]
        ax.plot(lts, vals, marker="o", label=var_name)
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSE")
    ax.set_title(f"{evaluated_model} – Surface RMSE vs GLORYS12")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No GLORYS results available for plotting.")


In [ ]:
# SSH RMSE across all reference datasets (altimetry + reanalysis)
ssh_vars = sorted(set(v for ref in scores for v in scores[ref] if "ssh" in v.lower() or "height" in v.lower()))

if ssh_vars and len(scores) > 1:
    fig, axes = plt.subplots(1, len(ssh_vars), figsize=(5 * len(ssh_vars), 4), squeeze=False)
    for ax, var_name in zip(axes[0], ssh_vars):
        for ref in sorted(scores):
            if var_name in scores[ref]:
                lts  = sorted(scores[ref][var_name].keys())
                vals = [scores[ref][var_name][lt] for lt in lts]
                ax.plot(lts, vals, marker=".", label=ref)
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel("RMSE")
        ax.set_title(var_name)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{evaluated_model} – SSH RMSE by reference dataset", y=1.02)
    plt.tight_layout()
    plt.show()


## 8. Prepare your leaderboard submission

To participate in the public leaderboard, two files produced by the evaluation
are required:

| File | Content |
|------|---------|
| `results_<model>.json` | Aggregated scores per variable and lead time |
| `results_<model>_per_bins.jsonl.gz` | Per-bin spatial RMSE maps (2°×2° grid) |


In [ ]:
results_dir   = OUTPUT_DIR / "results"
json_file     = results_dir / f"results_{model_name_lower}.json"
per_bins_file = results_dir / f"results_{model_name_lower}_per_bins.jsonl.gz"

print("Files required for leaderboard submission:\n")
for path in [json_file, per_bins_file]:
    exists = "✅" if path.exists() else "❌ (not found)"
    size   = f"({path.stat().st_size / 1024:.0f} KB)" if path.exists() else ""
    print(f"  {exists}  {path.name}  {size}")

if json_file.exists() and per_bins_file.exists():
    print("\n✅ Both files are ready. Follow the steps below to submit.")
else:
    print("\n⚠️  Run the evaluation (section 5) to generate these files.")


### How to submit

1. **Fork** the repository on GitHub:  
   <https://github.com/ocean-ai-data-challenges/dc1-emulating-global-ocean>

2. **Copy** both result files into `dc1/leaderboard_results/` in your fork.

3. **Open a Pull Request** against the `main` branch with a brief description
   of your model.

4. Once merged, your model will appear on the
   [public leaderboard](https://dc1-emulating-global-ocean.readthedocs.io/en/latest/content/leaderboard.html).

See the full [submission guide](https://dc1-emulating-global-ocean.readthedocs.io/en/latest/content/submissions.html)
for details.


## 9. Clean up (optional)

Remove the custom config and temporary prediction files to free disk space.


In [ ]:
# Uncomment the lines below to clean up
#
# custom_config_path.unlink(missing_ok=True)
# print(f"Removed custom config: {custom_config_path}")
#
# import shutil
# if PREDICTION_DIR.exists():
#     shutil.rmtree(PREDICTION_DIR)
#     print(f"Removed predictions: {PREDICTION_DIR}")
